# Batched key-padded attention with two vmaps

In JAX you can write a function for **one** example and let `jax.vmap` add the batch dimensions. Attention is a good test: write it for a single head of a single sequence, then map it over heads and over the batch.

Write `batched_attention(q, k, v, lengths)`:

- `q`: `(B, H, T, d)` queries; `k`, `v`: `(B, H, S, d)` keys and values. float32.
- `lengths`: `(B,)` int. Sequence `b` has `lengths[b]` real keys at positions `0 .. lengths[b] - 1`. The rest are padding. Every head of a sequence shares the same length, and `1 <= lengths[b] <= S`.

For one head with queries $Q$ `(T, d)`, keys $K$ and values $V$ `(S, d)` and length $n$:

$$O = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt d} + M\right) V, \qquad M_{ij} = \begin{cases} 0 & j < n \\ -\infty & j \ge n \end{cases}$$

Return `O` with shape `(B, H, T, d)`, float32.

Structure the code as a function `attend(q, k, v, n)` for **one** head (2-D arrays and a scalar length), and get the batched version from two nested `jax.vmap` calls. The length is per sequence, not per head, so the inner map must not map over it (`in_axes=(0, 0, 0, None)`).

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def batched_attention(q, k, v, lengths):
    """q (B, H, T, d), k/v (B, H, S, d), lengths (B,). Return (B, H, T, d)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[1.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 0.0], [0.0, 1.0], [9.0, 9.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 2.0], [3.0, 4.0], [100.0, 100.0]]]], "dtype": "float64"}, {"__nd__": [2], "dtype": "int64"}], "e": {"__nd__": [[[[1.6604769229888916, 2.6604769229888916]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[0.0, 0.0]]], [[[0.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 1.0], [1.0, 1.0], [1.0, 1.0]]], [[[1.0, 1.0], [1.0, 1.0], [1.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[0.0, 1.0], [2.0, 3.0], [4.0, 5.0]]], [[[6.0, 7.0], [8.0, 9.0], [10.0, 11.0]]]], "dtype": "float64"}, {"__nd__": [3, 1], "dtype": "int64"}], "e": {"__nd__": [[[[2.0, 3.0]]], [[[6.0, 7.0]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[-0.39530128858657, 0.2639148850157296, 0.6071282687955677, -0.9721597997272025, 0.7676642531398922, 0.25505812177433956, 0.7829798706300901, 0.27241823407236765], [1.1620081255305676, -0.9377563787668719, 1.776099733817743, 1.2023898441352874, -0.5999578484570015, 0.6601663130603562, 0.44476738448754977, -1.7457635656172505], [0.5952162456256633, -0.5853991209052885, -0.25001353358513434, -0.6023864391110364, -0.4280840067640782, 0.07173470176248886, 0.09671912405619901, -1.5591518413922796], [-0.26870603364939344, -1.3448111474077982, -1.2707778590591046, -0.346953600730839, 0.85530027587691, 0.6308565040330028, -0.6046580474158655, -0.7104208239983351]], [[-0.8274098970826956, 0.14274021925565974, 0.936650038779724, 0.01804985703847165, 0.6927666696591163, 0.3154090546513482, 1.5041856647552274, -2.0066907365561706], [-2.1310340831936396, -0.19843957784596147, 0.6365395187683154, -0.410549281758414, 0.42578068000505975, -1.173544797067956, -1.2593832343358193, -0.7165758868835466], [0.8413227181749481, 0.696358369654102, -0.713252712154986, 0.18111936994597383, -0.6475708354871698, 0.020059810497343504, -1.1142963805907677, -0.04655434751901477], [-0.735753934980961, -0.04435950785702231, -0.6520426060122302, -0.8217414802839252, 0.11343128787139657, -0.33779536082531136, 0.021671175782686587, -0.419085181552037]], [[0.6137428973723346, -0.8156847530495072, 0.7687510449895725, 1.331974309444846, -0.26888770649361554, 1.7452430767280316, 2.029698196865837, 0.4430750030154757], [-1.3576051491193484, -1.6156939484528159, -0.1757786841969012, 1.7759298264822208, 0.574549849965791, 0.7580309194582652, -0.2509133097904481, -0.7441468728010533], [0.37192110436980064, -1.1087787163755936, -0.19694438865759697, 0.511152295497356, 0.03593415340311622, 0.5245642290612524, -0.4459765954576025, 0.5981798358088446], [0.4046503807009621, -1.6497062609235813, -1.0422629538107075, -0.723476319045549, 0.6555863140819842, 0.3020801614586051, 0.2829882917455624, 0.17293243462011323]]], [[[0.6364764540263609, 0.9453478418566444, -0.55471618453936, -1.5909121646919766, 0.6673244554529724, -0.20942920166475995, 0.04880537673064059, -0.1738052649063213], [-1.1125682987307608, -1.5106513146782443, 0.9642801867742248, 1.0930925298253509, 0.9312396637512143, 0.20823464575150918, 0.2685207840712548, -0.4189296731440312], [0.068012235199191, -0.402836079808958, -0.5436861645760849, 0.20520316656139895, 1.698100442046864, -1.8835414686235434, 0.601511318239193, 0.6756512596372449], [-1.214406143769667, 0.31344994259797077, 0.10157496332371774, -0.4639827216892454, -0.4963243116217951, -1.4061731881369968, 1.707177778357255, 2.3475675133295377]], [[1.5863758938692298, -0.28779325388649285, 0.2856819811333672, 0.3772081316205152, -0.8129329524007229, -0.2585133595997185, 0.5332454085822927, -0.42162294678862666], [-1.7597561158297332, -1.2680186486126208, 0.25498837659016704, -0.3370613965236977, -0.2982936118044847, 1.3559778672193719, 0.08478662296430321, -0.7174362329016395], [-0.06496331660890184, 2.1579108218895597, -1.1264636491083797, 1.786948338595363, 1.441594990780037, -0.1600953058439699, -0.3358894071127189, 1.4188083086913799], [0.24397017291123366, -0.5308329934263428, 1.8227617267578917, 0.1744923763100515, 0.33214588311318144, 0.03851839966105425, -0.3013107249042945, 0.12420019946696831]], [[-0.3700959464536447, 0.5075311129071819, 1.0554538371570383, 1.085198198172033, 1.1913930981684038, 1.4954686333505183, 0.7610350163668245, -0.6044836257551762], [0.9621555437726979, -1.649206803175677, -0.7133713481582212, 0.28996866572309277, -1.6823936753767161, 1.4718807814018096, -1.017348199619935, -1.4978022370187938], [-0.7614302141268862, -0.7640095664566192, -0.026283924899902982, -0.15793461410276913, 1.0310812013462103, -0.16643926260653358, 0.17555119355907506, -0.9899791199757738], [0.497248770774718, -0.7749791139275669, -1.9482928215392434, 1.3038350841125743, -1.0564318047764338, -1.089701986610318, -0.8766761551128621, 0.6511177215972471]]]], "dtype": "float64"}, {"__nd__": [[[[-1.5234259916416433, -0.3981399491942163, 0.5768841319424516, 1.8969394746706036, -0.28968507982965547, 0.055826508337780724, -0.8672504362875534, 1.9812905890587145], [-0.14056637199013602, -0.06256305126712666, 0.14671833729993328, -0.9161278230111044, -0.01019480220174812, -1.2612591445422452, 1.4888421008879589, -0.1494730398924859], [-0.16486459210158966, 1.1681957582785087, -1.2158558848220218, 1.401256351044982, 1.5222818221033567, -0.7692632036460255, 0.690747539424684, -0.09757600098494713], [0.8827048205744942, 0.04195950390747259, -0.9001685960221104, 0.6281111109553829, -0.21847528930025628, 0.1792444521332749, 0.9764409903166166, 1.2152746518399424], [0.6291961792584582, -1.4211155964990294, 0.6944321886200565, -0.3005015865564779, 1.0722490582691877, -0.19584167337101632, 1.4259985863620415, 0.5140710669046085]], [[6.793543485522398e-05, 0.2932726544441482, -1.7639750667525753, 0.5738072281730505, 0.11494091340878991, -1.0967977188048088, 0.4070016227208897, 0.3192577348176515], [0.3576206242975661, -0.11539345708878417, 0.7885177058202848, 0.8014461611354436, 0.09359147673978564, -0.06111520178016605, -0.4537959833660088, -0.5950699624384738], [0.6117625185880927, -0.5458992774564005, -1.4122232079667902, 0.24595977005172406, 1.1074164771267676, -0.27360636966135565, 0.8682156166641356, 0.48149846923416645], [-1.0081942468903757, 0.32246976768228164, -0.10889124296772344, 0.682223880607984, -2.7856248635736574, 0.8312948280788758, 0.7414124616728882, -0.9247534906727276], [1.0697388192933963, -0.15130722249107023, 0.1845022021434689, 0.8662132063215051, 1.3465386087585518, 0.17360481455514865, 0.688291577557406, 0.4462666827414231]], [[-0.7248194388554418, 1.6449662232519746, 0.7813403082745828, 0.156700131611198, 1.083947940445542, 0.5539845188708896, -0.5421837845932549, 1.0166933918112726], [0.0784588327669846, 0.6362290891024897, 0.9726846896837326, 0.9654582584990465, 1.8991630189112358, 0.21829731730818147, -0.1466890000518128, -0.7031407553053078], [-0.6027897264430842, 0.6930890952459619, 0.4513877187669522, -0.8346272758764623, -0.5825162357253724, -0.4084715404896509, 1.5894499115276843, -0.8972784619763616], [0.39727334202109627, -1.5898148267160814, -0.19121954630797752, 1.42350731339307, -1.2102336016678912, 1.1903412349091838, -1.8462027198509605, 0.09528207839182529], [-1.063329287546252, 0.01901480233312343, -0.20140987821544276, 0.8510774053440736, 0.3089255698567089, 0.2063516130471289, -0.5036148267046537, 0.4865593554618768]]], [[[-0.7586924235367061, -0.6019229252180535, -0.5874203475439159, 0.6054588380533384, 2.298340994932344, -0.025761464349973624, 1.9574046479745844, -0.9767245878111889], [-0.5579163007463475, 0.5591204937058211, 0.6481375284198518, 0.8010694444716239, 1.6163401231181, -0.16196657693915434, 1.124964555390539, 0.290257705909035], [-2.780587856639197, -0.9478462610353828, -0.48086995513114983, 0.1566597759404357, 0.5753792545869604, -0.228369719056136, 1.3561969295906768, -0.9229505698673857], [0.021755413254581496, 0.9650007879542244, -1.075555449216515, 1.3008931319604966, 0.9983919324818832, -0.11492114694347667, 0.05099400573225651, -0.040401233830665594], [0.06839208673797367, 1.2911578682388358, 0.5613875882114832, -1.3276869360720291, -1.4929348133812654, 0.6474104502169915, -2.5776150647263334, -1.5289541262487212]], [[-0.5708026153496051, 0.016263792020536745, 1.3869352166445277, 0.21359480860371988, 3.543037704854227, 0.2497892779874184, 0.8905138867723432, 1.0490736733613784], [-0.07181970439759625, 0.9846828853544083, 0.6284646320874582, 0.26901647976402876, -0.5988590612920697, -0.39397434228861405, 0.8278662777624761, -0.0032894717200770427], [0.43514842476537907, 0.43553125634532486, 0.31011626601262127, 0.04715470467760803, 0.619000989644874, 0.8614473173742426, 1.4039926408672414, 0.5322123238340821], [-0.5289781911722888, -0.6422201331945767, 0.6691192219994475, -1.4880560336530309, -0.485151456671021, 1.070965457225669, -0.29628428165655657, -0.6187967335494434], [1.2623355436372012, -2.613171741623092, -0.04304455344124896, -0.42336415922630627, -0.4543963707910916, 0.9381310352439323, 1.5284555439125136, 1.1630782120495018]], [[0.6534404883193268, 0.35796645765382007, -0.20916732233746632, 0.03651710553388239, -1.5834389250723568, -0.48761777182063926, 0.6277426891564911, -0.08652953384541183], [0.8559592052907321, -1.5165857769098154, -0.5270207440038213, -0.9652652774319342, 1.0826169676052821, -0.22852941708183167, -0.2593539245473229, -1.1851441572187045], [-0.4682269993858051, -0.9727950999803793, -0.5356855045541156, 2.197944909271107, -1.1169816122467204, -0.2275369056493768, -0.8591797430809386, 1.3312807167846197], [-0.9384475978636716, -0.8350302356380308, 0.3929015548636508, 1.6792041514919482, 0.25836071689685364, -0.5410174522914828, -2.520631018613691, -3.584464584498713], [-0.05720273081033771, 0.04819969234298533, 1.199306119249865, 0.4874436319125008, -0.2391422732717059, 1.2977532333043655, 1.7428299371830858, -0.6324559559158519]]]], "dtype": "float64"}, {"__nd__": [[[[0.9449775759912037, 1.2103668614985865, -0.7433684940829819, 0.5822667371140716, -0.013637988511873488, 1.1899148918032372, 1.1226704142190722, -2.099175415769622], [-0.20800226228135257, 1.3876175613040804, 0.5265130959327202, -1.008304367161351, -0.4625836415532583, -1.05281223910202, 0.34350706919125823, -1.2836406875341577], [1.2366054731858596, 0.863166544102697, 1.6956836512626114, 1.0935087058739794, 1.038077645954742, 0.9365491260557999, 0.5971190350755541, -1.9021104848480053], [0.5834382685662215, -0.7524120456860055, -2.3580312569655235, 1.6068954466233776, 0.0261579816083452, -1.58937290345735, 0.3385200945492352, 0.4669429496197702], [0.22740769110188155, 1.5045219792716589, -1.0760059601468912, 0.932050450373597, -0.9419798968979491, 0.5046000993693522, 0.6818916000021091, -0.4501282150583147]], [[2.553538163324132, -1.7746614690751428, -1.4509872313625893, -0.4346076495334098, -0.8626020675591859, 1.1667429918297663, -0.5529672845815184, 1.3640046319118677], [-0.4121207900177814, 0.8367991478042597, 0.8300655081331314, 0.146605675933901, 0.7459577304500747, 0.3809039562758178, 2.1447653177968777, 2.4106515492974823], [-0.30921291727724054, 0.5258227658197887, 1.1899629680001662, 0.6051384647338347, 2.0072734778324866, 2.6779760585134786, -1.9439350275835687, -0.5113289297867515], [1.3721484390789787, 0.5701846480703028, -0.6720284032871408, -0.19949554983507, 0.1727953592835805, 1.6986084401054489, -0.9121340827576457, 0.25490677825520874], [1.5241605550241053, 1.0937745243698704, -0.09314205986075712, -0.23793413086278203, -1.7494406854297933, -1.6729521778675704, -0.8457815746924643, 0.8755425083978499]], [[0.8541547172219084, 1.3367604909133983, 0.9500565669378069, 0.7979133300901686, 0.20529170332748603, -0.7635186916406551, 1.2554240331280655, -0.3438671051240644], [-1.783871765990227, 0.4823305265031308, 0.12101549850619843, 2.4667028231800514, -0.23714642821145163, 0.38016489382988733, -1.12817795226139, 0.7105526317732384], [0.2889063783008983, -0.7916283006828269, 0.26421032693377206, -1.2588024784373084, 0.5971057381299691, -0.4445310619618362, 0.3746279375311829, -0.8981724941457758], [-0.7606667761236254, -0.15165750206219078, 0.846519787982925, 0.14029012424086149, -2.373347930114152, 0.679352534929921, -0.10356658136067173, 0.6440686888741336], [1.1608427705895605, 0.8357692788865869, -0.7258838536040022, -0.3577231902540261, 0.5077568908328721, 1.0190932544083882, -0.7686432792448947, -0.6319220577510385]]], [[[-0.9879582319720063, 0.41138998064590165, -0.7186739256029496, 2.3346049954224726, 0.8425583759574132, 1.195311401854755, 0.6160955250669505, -0.5910398391511055], [-2.5523479950694, 1.4707382174243644, 0.3168754657511549, -1.89251174577699, 2.857417308398872, 0.759595883493327, 1.6143240839550215, -0.23962127741574016], [-1.773524406164664, -0.7770264193549921, 0.8052572197807112, -2.1145687683611256, -0.5269166797239874, 1.1634153363788853, 1.0872115590057068, 0.08793649372992199], [-2.569815007014502, -0.2549367243839244, 1.303741995108828, 0.8854002368207518, -0.38818820894617545, 0.3908516033786845, 0.5884590854717898, -0.3863752266147558], [-0.7310280445011604, -0.6946631583986302, -0.8359198746921428, -0.0668886643421378, -0.030821338920872098, 0.5965975803542072, 0.6009596523423841, -0.12646372351796983]], [[-2.1368574949812524, 0.5723321309744546, -0.8873356654487232, 1.7576424746370187, -0.2896389613543849, 2.2650321029177523, 0.6562109503422947, 1.6972239036574002], [0.21481393203045368, 0.5325136636864569, -0.40745078122867684, -0.45582070811583525, -0.4552024117544363, 0.7678586759290762, -1.111286646322974, 1.0416642592520229], [-0.24414706728703633, 0.6470335250769098, -0.9680948851454563, 0.07737495160479457, -1.694391985059433, -0.631376215399936, -0.18627436908099249, -1.0059627847383756], [-0.22322685693590713, 0.12271004525398393, 0.14007771582352174, -2.0414699690841194, -0.3008371133955535, -0.2119577842665349, -0.8498441404014584, -0.9611077560967894], [1.9240530983498008, 0.8394057773816513, 0.9947871939776144, 1.195285382059803, -0.9367231568748363, -0.2560857626918455, -1.7697302468301153, 0.18869368941719586]], [[-1.9147093004865956, -0.7230841830650089, 0.744665933513062, 0.5401005529965468, -0.9814415212539587, -0.8986512872504236, 1.3031267081215894, 0.445946892760851], [-0.5042356567469672, 0.1442878072600871, 0.12388610467600215, -1.0764393343449736, -0.5541423916282453, -2.105225520016432, 0.238083436191829, 0.08277342984905878], [1.0373445120611795, 1.029620992798984, -0.45418041381695634, 1.600964614199409, -0.7315324396221344, 0.9397301517648157, -0.5193395909052296, -0.6208443329521166], [-1.2433080161016339, -0.062287563022193584, 2.2565703190412596, 0.15589439387235776, 0.4730592957333231, -0.33943750041431664, 0.5049340947463096, 0.6251394508608439], [0.47841517359771607, 1.376749516495455, -0.02247784141705673, 0.421090834154706, 1.5518391975153933, 0.6295466715635412, -0.744873033074644, 1.2812359917653167]]]], "dtype": "float64"}, {"__nd__": [5, 2], "dtype": "int64"}], "e": {"__nd__": [[[[0.3592861294746399, 1.0550094842910767, -0.33542197942733765, 0.4321151375770569, -0.29823023080825806, -0.10233261436223984, 0.5672754645347595, -0.9479123950004578], [0.3871798813343048, 1.0535180568695068, -0.6952018141746521, 0.6647588014602661, -0.41767603158950806, 0.04907326400279999, 0.6206181049346924, -0.8002775311470032], [0.31118735671043396, 0.9727560877799988, -0.3049846887588501, 0.3593546152114868, -0.2768380045890808, -0.3144474923610687, 0.5107374787330627, -0.878923237323761], [0.5693156123161316, 0.8930203914642334, -0.34420332312583923, 0.7493940591812134, -0.09045906364917755, 0.10216923803091049, 0.6118274927139282, -0.9960435032844543]], [[0.8670420050621033, 0.5282887816429138, -0.034922897815704346, -0.044593822211027145, 0.07268834114074707, 0.798721194267273, -0.2608550190925598, 0.8954458236694336], [0.8317447304725647, 0.19456295669078827, -0.05778857320547104, -0.045518726110458374, 0.17120321094989777, 0.9134175777435303, 0.1456204503774643, 1.2361280918121338], [1.0791471004486084, 0.06596305221319199, -0.20009495317935944, -0.07115571200847626, 0.03230762854218483, 0.9678242206573486, -0.33705785870552063, 0.9625332951545715], [1.0871715545654297, -0.028229326009750366, -0.1811857670545578, -0.033617760986089706, 0.16169169545173645, 1.199188232421875, -0.6157900094985962, 0.7717320919036865]], [[-0.3029710650444031, 0.176545649766922, 0.35384678840637207, 0.3667791783809662, -0.6046856641769409, 0.2787341773509979, -0.17277707159519196, 0.05164071172475815], [-0.36324453353881836, 0.248601034283638, 0.33424538373947144, 0.4156187176704407, -1.0754642486572266, 0.5884882211685181, -0.3489713966846466, 0.24676068127155304], [-0.3437378704547882, 0.19782686233520508, 0.48030877113342285, 0.30873653292655945, -1.2635183334350586, 0.5159856677055359, -0.17980341613292694, 0.26823633909225464], [-0.19418656826019287, 0.18914927542209625, 0.3183690309524536, 0.2278522551059723, -0.724385678768158, 0.3939054012298584, -0.20066457986831665, 0.04678856208920479]]], [[[-1.7068750858306885, 0.8982145190238953, -0.24278613924980164, 0.39202916622161865, 1.768488883972168, 0.9950778484344482, 1.0748324394226074, -0.4295450747013092], [-1.5058238506317139, 0.7620698809623718, -0.37587225437164307, 0.9352871775627136, 1.5095444917678833, 1.0510749816894531, 0.9465427994728088, -0.4747084677219391], [-1.5505603551864624, 0.7923638224601746, -0.34625884890556335, 0.814405083656311, 1.5671629905700684, 1.0386147499084473, 0.975088894367218, -0.4646590054035187], [-2.0644235610961914, 1.1403334140777588, -0.006106539163738489, -0.5740979313850403, 2.228994131088257, 0.8954930305480957, 1.302982211112976, -0.349226713180542]], [[-0.21193207800388336, 0.5397393703460693, -0.49453309178352356, -0.0541546531021595, -0.42515844106674194, 1.0395432710647583, -0.7905477285385132, 1.1606253385543823], [-1.2079473733901978, 0.5566039085388184, -0.6977812051773071, 0.8833246231079102, -0.35503655672073364, 1.6736485958099365, -0.04195062443614006, 1.4382779598236084], [-1.718477725982666, 0.5652481913566589, -0.8019606471061707, 1.3638509511947632, -0.3190939724445343, 1.9986737966537476, 0.34175992012023926, 1.5805951356887817], [-1.5750669240951538, 0.5628198981285095, -0.7726959586143494, 1.2288683652877808, -0.3291904032230377, 1.9073724746704102, 0.23397356271743774, 1.5406172275543213]], [[-1.0735820531845093, -0.20583227276802063, 0.3744677007198334, -0.42391303181648254, -0.7266243100166321, -1.6181844472885132, 0.6679947376251221, 0.22937065362930298], [-1.0075052976608276, -0.16519828140735626, 0.345385879278183, -0.4996434450149536, -0.7066065073013306, -1.6747090816497803, 0.6181004047393799, 0.21235696971416473], [-0.7048570513725281, 0.02091553993523121, 0.212183877825737, -0.8465078473091125, -0.6149201393127441, -1.9336062669754028, 0.3895718455314636, 0.13443008065223694], [-1.4674628973007202, -0.4480496644973755, 0.5478231310844421, 0.027512775734066963, -0.8459494113922119, -1.2812433242797852, 0.9654126763343811, 0.33078840374946594]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[0.6465003541509388, -0.7998770929130626, -0.7727623167052589, -0.385778288483983]], [[0.8346342537176381, 0.34398818053744207, 0.9958153414776598, 2.479432528149894]]], [[[1.7478473766274505, 0.12441655393012985, -0.6500998515586293, 1.4072615906482988]], [[-0.6940190793605724, -0.7654364794154525, -0.10100942345862418, 0.2969019336945648]]], [[[-0.24168119461517854, 0.279380805505678, -2.6746254338319773, -0.354189152483647]], [[0.04927453624028841, 0.7242475937078926, 0.3641577383939416, -0.21119356540906664]]]], "dtype": "float64"}, {"__nd__": [[[[2.5104085388288295, 0.9999806218506042, 0.6375816664766865, 0.3313914640665949], [0.4794115165694021, -0.8001515926358916, -1.0052053278775268, 0.24999484923315177], [-1.5738491665960421, 0.6134695820327016, -0.3702804636780358, 0.7263396003394431], [0.13713262321559044, -0.8094256187136643, -0.6870294294854603, -0.39463916176052655], [1.3211604023375252, 1.3954188823827685, -0.1709428912111078, 0.15117396876755984], [-0.7797881297660109, 1.429650418295845, 0.2601883128225055, -0.6279731275156257]], [[-1.292043855148919, 0.5351314314458022, 1.8897226956666475, 0.10749458038354212], [-0.8788115637998645, -0.8536117347524433, -0.5556792812706723, -2.260111761949279], [0.9099362859544923, -0.5607042237555494, -1.8212615775592265, -2.926874734308934], [0.3450079891083276, 0.2146378170453011, 1.4253186459777667, 0.9984395451466545], [-0.40859561737359373, -1.7188009976907643, 0.20790106700523106, -0.5753607617250147], [0.08453218400170862, -1.5889386808139505, -0.2998954517436731, -0.22755072399613424]]], [[[-1.283217553200469, -1.7526430510042716, -2.219750228925221, 0.07247970688629665], [1.2008987867033412, -0.040300674122441815, -2.5802582781929226, -0.05143554329586471], [-1.680045755516664, 0.23618083753989433, 1.7267365806986643, 0.3454809186784457], [-2.591899703406305, 1.0434793988167408, 0.3280613657146427, 1.879842226465557], [0.4997894314003217, 0.6902827675656585, -1.7990196782703427, 1.4237289350532232], [-1.2426914349454001, -2.0181746741518585, 0.5033663105602847, 0.7265804556364263]], [[1.4438488160065102, 1.150761172516862, 0.3670379913146726, 0.32908592176530604], [0.8603412844275949, 0.35934245544530813, 0.45740058267166195, -2.033700302091372], [-0.1102357889690413, -0.3684223329941079, 0.5084225616734619, -0.7955745828831363], [0.9359046165693189, 0.7447285531907063, -1.126472963561533, 0.0255204167573806], [0.5436481209762465, -1.0848682607210127, -2.4155594129237383, -0.25002182386202443], [0.24441215765276597, 0.01719907078109228, 1.0841052730677156, 0.29803768471951947]]], [[[1.6115491343224655, -0.5703754124625151, 1.6535920401972521, -0.2910756623478765], [0.5009311778232839, -1.88031415221228, -0.7762830078668634, 1.1902472814100071], [0.06678732006881659, 0.010361981123264094, -1.208250543920799, 0.7952855340090841], [-0.9990519988090435, 1.4648137767815526, 0.6153947701106753, -0.5622377920685345], [-0.8193257105040831, -0.5262561410330757, 0.3781804974056244, 0.2504798117033315], [-1.6195585973912878, 0.3478673535272846, -1.0747138024018115, 1.8406562348247884]], [[-1.8914342171463479, 0.9392959418065416, -1.1971539817170216, 0.38289396053680635], [1.185055862514284, 0.4724362744593919, -1.178102535057472, -1.5447271859784517], [1.4815179472427271, -0.8105404612870419, 0.7845354017799773, 1.200627724249146], [-1.660945455860599, 1.9807394584952436, -0.12733219017309388, -0.1987749085119995], [-0.9882063638218987, 0.23311442292485554, 0.4838526627432238, 1.0569445116943958], [1.043334803901286, 1.032485255803081, -0.11037300929720532, -0.09188999026702646]]]], "dtype": "float64"}, {"__nd__": [[[[0.129518164059734, 0.19163034883291688, 1.2744083673781617, 0.9260583606071836], [-0.1622213991288348, -0.9224324704095341, -1.9160601671917197, -0.2179595190052761], [0.019501536777801212, -1.8780906228528889, -0.3607489942302511, 0.2546367651926883], [0.21745381898090374, -1.4168402641282078, -0.6711909320851933, -1.4620905214302164], [0.8534194478363153, 0.27782843384088657, 2.608417424033557, -0.6026420813766306], [1.5243656802322785, 0.03268630346316836, 0.08564427883978502, 1.0573983354723775]], [[-1.9373326164042046, -1.197599108410216, 1.5261445982916175, 1.006951591794048], [1.8757092617658273, 0.5658403384579376, -0.47745105185448605, 0.8596386488851683], [-0.4365011184709154, -0.06746035638836222, -0.12113280095875865, -0.8670384838025453], [2.1249021491611546, 0.7252494729098645, 0.13607437329539981, 0.39601664727769637], [-0.5698494879995372, -0.3717188541281438, 0.12299338861532019, 0.326670260898289], [-2.171913989216002, 1.3202437062671224, -2.05628223217485, -0.9680041181893042]]], [[[-1.397643571376839, 0.10635692685900042, -1.7719594310449767, 0.9893237322264471], [0.6910830133349065, 0.9293277485114181, 0.7979755662005868, -0.3330443695391465], [0.24175014787199336, -0.9690763795030434, -0.44009229697547997, 0.4315586828344365], [-1.2734567421587975, -0.3043396931343326, -0.5072780226067014, 0.888058038421329], [0.528890499331527, -0.44257780972610183, 0.7964950953781323, 1.93885430526012], [1.0962768560811984, 0.09337967993416428, 0.12725476349539053, -0.9732166847549512]], [[-0.5435022552752655, 0.2790743940640189, 0.37717155003502356, -1.4315332282300859], [-1.2522229864959438, -1.2332902556568932, -1.160212553212622, -0.6441863247428445], [-0.1684529558712091, -1.1936998964048549, -0.712463864491933, -1.9948708820726033], [-0.25393848499777083, 0.21536522037906206, -0.5748697964764512, -1.5984984164213492], [2.365990606731273, -0.8623559851270041, 0.3870520469481592, -2.2241651510804012], [-0.7924401978499218, 0.09402283601408702, -0.1259334787787657, 0.1044597005573307]]], [[[0.0061616249272874015, -0.7984717464967491, -0.9546368728187462, -0.2424338047067513], [-0.7110884069965593, -0.6584661084778122, 0.34872286951146664, 0.16605930694562318], [-0.5205350499838012, 1.016300735850961, -0.13195393030005453, -0.30937438682370055], [0.7609576560230281, 0.18155052651666947, -1.6489514669105452, 0.39829480575756293], [-1.6703998980475099, -0.3820629245817418, 0.22414603061827293, 0.5866102541401338], [-0.9699174947992621, -0.43092833166272587, 0.7094201925603517, 1.947117186213516]], [[1.9028193196518943, 0.7833452155415035, 1.6065279647331376, -0.5948908742038963], [-0.27957316547319194, 0.7550939388808945, 0.754746632737065, -1.5758415789486626], [1.1248247204354844, -0.9019490080486491, -0.15830595377055134, -1.1974649159159896], [-0.39931557395940787, 1.0160383928565953, -1.3385413997300157, -0.679069124575758], [-0.03853380822681345, 0.5473030453334571, 1.6678781533301827, 0.0876980277192774], [1.0137736043315764, -1.5906102166902147, -0.5659031694951211, -2.4218908017297385]]]], "dtype": "float64"}, {"__nd__": [1, 6, 3], "dtype": "int64"}], "e": {"__nd__": [[[[0.12951816618442535, 0.19163034856319427, 1.2744083404541016, 0.9260583519935608]], [[-1.9373326301574707, -1.1975990533828735, 1.5261446237564087, 1.0069515705108643]]], [[[0.48655441403388977, 0.14140552282333374, 0.632770299911499, 0.8714087605476379]], [[0.2925739586353302, -0.5617210268974304, -0.22010357677936554, -1.4330775737762451]]], [[[-0.5647899508476257, 0.5345151424407959, -0.01303114090114832, -0.179062157869339]], [[0.84850013256073, 0.328233003616333, 0.8099708557128906, -1.1354304552078247]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[1.1161199166099551, -0.614549808368069, 0.5712592565637973, -1.0931911568073527, -1.1368608645697462, -1.048568372653241, -0.7498426205113413, -0.34458666755639156, -1.757562363517266, 0.04141114057346206, 0.5199729630846994, -1.8618753005042779, -1.5957642578297386, -1.1977493804808463, -0.9671710578901161, 1.6798584824650828], [0.6902992802034954, -1.3196817472099132, 2.6630247804916247, 0.7475103642859577, -1.3248031854304643, -1.7433028664838883, 0.3325316897348787, -1.7233795748264051, 0.1028274963884664, 0.8172793622339997, -0.9056351929877333, -0.8009444985876796, -0.8977340504304309, 1.248950470051646, 0.5956138732810007, 0.4380834983063395], [0.7920514098693098, -0.07773007062688352, -0.6983513261145139, 0.26398984905884176, 0.8653379325929113, -0.2012725992341848, -0.26872261437687783, -0.993010742155404, 1.3529074523710904, 0.7305850872426073, 0.1755749620660547, 0.16556746884512322, 0.35261823369392864, 0.062177038606075194, 0.9667825274241206, 0.17878633725371304]], [[0.7529708740304485, 0.05850669759278116, -0.5802467473706192, 1.1388977852097697, 1.1667719134293832, 1.5389352664965792, -0.42778939043255515, -0.6329684000228946, -1.8614089462060257, 0.0564744479224794, -0.82687587257564, 1.2008172593568482, -1.1632015418470414, 0.9568781759741899, -1.236431234612107, -0.11771770071134324], [1.2709680381866277, -0.471494778766695, 0.3901189187426041, 0.7021298225490307, -1.3497400502203294, 2.259987814657551, -1.3521644604778813, -0.2984661938876075, 0.199184728477823, 0.4110038469854155, 1.0057316187885523, 1.1937477857183, -1.3408817030876246, -0.168388977934628, -2.079114766315723, 0.27576051570974885], [-0.6715865593517752, 0.3488396950984449, -1.7127936410170548, -1.128907704418855, 0.49591367181409957, 0.7209325004591691, -0.3258004726050034, -0.45785468376101324, -0.5625005944024174, -0.5895718692074589, -0.6080636261240562, 2.5658038476410847, 0.7130829855245671, -1.602407177147185, -0.4335849024414317, -1.024545398676068]], [[0.15823616267436535, 2.1363378080038813, 1.2425088840159777, 0.1443131342317201, 0.40053714301441495, -0.12533962906902008, -1.0109377178488355, -0.5975384187082088, 1.111434234327399, -1.5791316834582236, 1.276073767332804, 2.3408936971975436, 0.07145849398876426, 1.1378807734979153, 0.19331865878684149, -0.27564772282474215], [-2.0372428876374236, -0.7284642920570485, 1.5592099876388383, 0.08894914321611781, -0.3102053345361235, 1.5832280472198963, -0.06300258885462949, -1.960612023977907, -0.5291662669247973, 1.5230521043139744, 1.1610426723733518, -0.3160720145307965, 0.8403628103288582, -0.0982763676824906, 2.4841860518009993, 0.7708772606825658], [1.2341911487218096, 0.5275670751156154, 1.1437045473371366, 0.7432397248004905, 0.1699018552063201, -0.374822966388521, -0.267007583288147, -0.04444134802435802, 0.854521462232738, 0.7438057004228797, 1.489354201562345, -0.08204462397248176, 0.2815809496586738, -2.010912862442713, -0.6208733073846089, -0.5788090526246877]], [[0.06847189439621418, 0.1984790886764798, -0.419419338853488, 0.45349526534651025, -0.4088594420791236, 0.036377196621995025, -0.5812825843680617, 0.2390935162667082, -0.55112411793365, -0.18548189292024142, -0.7710667772213255, -1.3114919148453417, 1.3195737320270948, -0.734695633706047, 1.086587029574075, -0.32012487441570087], [-0.01523270015742574, 0.08928790510948416, -0.8618122468217356, -1.6927355487847795, 0.7885525689131659, 0.22052111863758675, -1.6367211932913648, -0.722647247663025, -0.636512166326599, 1.0886918384290867, 0.5133900082408831, -0.01881019569544778, 1.1033525595245248, 0.40729248507422267, -0.7357279607003794, -1.3951853663945017], [0.9713966584553237, -1.7120463213590074, 1.1864358794489687, -1.7992291380028953, -0.6686965130924313, 1.2110430881573544, 0.8755678638898989, 1.159079924691211, -1.4996264271388098, -0.31722262517587657, -0.07410409636495607, -0.42930421414097414, -0.6066574807794647, 0.2764333684827065, -0.010005903631272248, -0.9770870229078027]]]], "dtype": "float64"}, {"__nd__": [[[[-0.5885673732471131, -0.2868419890280369, -0.05276638508184126, -0.2821976636757304, -1.5538517137595116, 0.06785271620011128, 0.48143973833325276, 1.5541962036890802, -0.5245168691484506, 0.8973386996722865, -0.5631275152364484, 1.2553338385689345, -0.13288670079236248, 0.1035197687378352, -0.20726089519401156, 0.16131541035010263], [0.19005785545622486, -1.3765336276133957, 0.6732778053066434, -1.006792207397219, -0.16962914312888663, 0.9558424983237768, -0.2040505827676341, -0.6415495544887297, 1.6983895890089555, 1.419087977083333, 1.4384448126519154, 1.071881191868305, 0.5219932960627357, 0.2401784639368252, -1.4156137774279185, -0.9019532481902048], [-0.823109542293424, 1.1121766701921645, 2.443997042739468, 0.8278530456395067, -0.17042368733902125, 1.4081063659190636, -0.03601275903841946, -0.20406494239846176, -0.04072648953727337, 1.1453052370686803, 1.2806695995940702, 0.3293428737637492, -1.0143349480199986, 0.8835310084108388, -0.7706427038168909, -0.9020080582332819]], [[-0.5607842209122155, -1.0693374615820233, -0.7160670485201207, 1.7539781675698645, -0.9805360525721891, -0.07138762211610582, -1.5117987599789917, -0.8210425259395369, -0.3673944008111522, 0.10949775128353506, 0.10488089927082787, -0.07412540775364629, -0.7119231439541588, 1.4907256590754703, 0.7163267171956376, -0.54946667323207], [-0.2410743418563942, -0.4556986921622119, -0.022792334610257616, -0.6422622496809199, 0.7384717807510331, 0.9447696663207682, -0.12790686013930494, 0.36719992082120595, -0.5758433971290903, 1.2699543649685072, -0.3965204957517266, 0.5071042912717115, 0.3738618025748671, -0.9333222696968025, -1.8182197108312215, -0.7620167983746037], [-1.05737569431666, -0.6522806742425077, -0.08684450738583024, -0.09038155930737087, -0.5495466681101154, 1.73651216719364, -1.2332539480465146, 0.9479604801411549, 0.7367310192047627, -0.8318459720150371, 2.0073865467691245, 1.325876438383178, 0.33327996837410345, 0.4792458783281507, 0.0747831556808442, 0.638061974371549]], [[-0.49725735878756405, 1.171066003025492, 1.0253073833091102, -0.3666056895994363, -1.4855818443364244, 1.1606672480586198, 0.20228974118895177, -0.8818291015721701, 1.4737221205904247, 0.22606192594739763, -1.51694105747588, 0.9616043355896895, -1.163485480779385, -1.9017298161712581, -0.7434409077999623, -0.21291783510972606], [-1.886443326527824, 2.367124765175459, 0.28997206805296133, 0.20578744108474134, -0.20330672598825542, 0.29932500970913906, -0.8330297959500145, -0.28392890936505927, 1.1698709245398362, -0.28627403040889815, 0.22469429165852142, -0.21652946342725998, 1.9224507760726004, 1.232159101147373, 0.44506138996663647, 0.7926515703371114], [0.3779157661624318, 0.33248457419551997, 2.796947479361707, -2.240839229433677, -0.14594390869866758, 0.06401022011865672, 0.9883536651168644, -1.5170617521233758, -0.6423249269463002, 0.42779352410634763, -1.9352651316938136, 0.5749775719812975, 2.235380165747175, -0.9469984159552619, -2.065579594791625, 0.7013248802538553]], [[-0.27445994467934015, 2.1439090886161565, -0.39855459920853614, -0.6417159856605473, 1.2213024570487783, 1.3317906173417153, 1.136324805238341, 2.3492887074956728, 0.6250838552801316, 0.07349542764647515, -0.5230858406696879, -0.10912201092299516, -0.4528889671288923, -0.5412031460831435, -1.077883529454664, -2.3889916409087437], [0.4621421703090596, 0.43715097660004604, -1.709948833813063, 0.9079504799284246, -2.837753027496245, -0.5012834937665986, 0.15805054572986635, -0.35745252773559355, 0.766371178404317, 1.2538656722917585, 1.242723022352896, 0.1345697184886932, 1.6645251772368201, -0.365093230108852, -0.5505801855746824, -0.6419580545294038], [-0.7297184995629413, -0.41791267008065996, -1.3465346402393568, -1.6593711500936899, 1.5958982418579906, 0.6155575129890721, 1.017503100145363, 0.3708692156859669, -1.0681430889756052, 1.3424733694765065, 0.7730174876136504, -1.2274308175259976, 0.7871042362541492, -2.1277594123058563, 1.047837484849309, -0.8636338774175168]]]], "dtype": "float64"}, {"__nd__": [[[[1.0577000790686373, -0.8564956180173111, 1.111659195015263, 0.3821620433371548, 0.6298742773814986, 0.30002834495367, 0.5550927933213436, 0.7000190031611417, 0.8188809010995011, -2.3078105761748358, -0.38609241338870526, -0.5445693096268714, -0.30700269415025505, -0.1408189727292095, 2.0682864585046623, 0.5758486483265407], [-1.4435968405922615, -0.45869816838797844, 1.0014401904109291, -1.1170812864803472, 0.2388631477355397, -0.5986392999035842, 0.3702671718337582, -0.20828801224772792, -0.03453323284439828, -0.2704053567953051, -0.8501878920611915, 1.276386226375912, 0.39090199988443847, 0.12712124008031264, 1.5881448261690376, 0.28787340094372876], [-0.030354519382520838, 0.2229851100322054, -0.3985586934264266, -0.8161458071431272, 1.8946108335374587, -0.8175868858887977, 1.2784893314434944, 0.4841803400655134, -1.1739937041109871, -0.24296859238411567, -0.7764348831679989, -0.06656701924081515, 0.8569794376746929, -0.9269225509569374, -0.9074987945212891, -1.2219282965412843]], [[1.49829320988817, 0.12948356973258218, 0.348131806257394, 0.1666726942818596, -1.169100088632123, 1.6975350999712602, -1.1419539903496634, -2.2389129655128266, 0.7431965815361449, -0.9292703231543723, 1.0217982120867033, 0.714446998419309, 0.11888036112678321, 0.6983495433374355, 0.7700330847799983, -0.5231799707212019], [-0.788051031057033, -1.3292905447500956, -0.4318771020876893, -0.7718830761690199, -1.1236204381635706, -0.7599384105782121, 2.5893547564360917, 0.6511883106931073, -1.4156911124120977, 1.6827159671822534, -1.860189805707008, -0.13632102429666762, -1.1217654445542358, -1.1081071166605663, -0.5309687918024241, 0.47338326603787434], [1.148593193720416, -1.8978132810521477, -0.4746876402832556, -0.8953533847148855, 0.5254103505535134, 0.8141991689638061, -1.3237956608184045, -0.8701513978424825, -0.8324870378305714, 0.7285921682331701, -0.4955966070493806, -0.8622303602217778, -0.7319138949036001, -0.7962369610024873, 0.07172638777092552, 1.7861484260838556]], [[-1.8168345844535176, -0.7733543716426904, 0.3606375529222824, -2.321490012763601, 0.3697643204861112, 0.369934587062281, -0.004419848667784356, -1.419907558963328, 1.797398322580339, 0.1691612891466318, -0.29413108077468936, 1.1454648041778372, -1.6133061639778015, -0.835026146823305, -1.9740847811178033, -0.2379534624724956], [0.07282033198681871, -0.3709206567895686, -0.28996206445946693, 0.652844389074244, -0.7322347961106702, -0.35075896919461635, -1.36702769592941, 0.5532201210841057, -0.3557664137634043, 1.6331441508394255, -0.6340839067936302, 0.5070893050616154, 0.3408569478613927, -1.3321740712325243, 0.7329670889928267, 0.46751737071869454], [0.9785704402875571, 0.8971565068719819, -0.31600745191078794, 1.5839923839027317, 0.4033276727429383, 0.7881970352216455, -2.0838770455174562, -2.005867759663553, -1.8160292077799531, -0.11146848687515268, -0.020983302630666344, 0.5927072568502714, 0.06871888603773772, -1.7125850380205159, -0.4834531216839933, 1.4651403877121951]], [[1.0944557124755407, -1.6030099687949473, -0.3279179978254454, -1.4845526601520274, 0.3901753316101487, 1.0284421476008767, 0.8197654626668923, -1.103458390787911, -0.5302280372711695, -0.16193526655633067, 1.7558611010583345, 0.3010441009582117, 0.11188463620279981, 0.13028736420016673, 0.21534181949285655, -0.8566217057575849], [-1.1431985291971511, -0.5524164743101478, -0.3063527426359549, -0.652312750988315, 0.7024654926408662, 0.01721538109292966, 0.7682072538867702, -0.7387204036051169, -2.5023993146346126, -0.5335042678939145, 0.8856715686165506, 0.18127025084659684, -2.59322501811646, -0.583008142687885, -0.7456207622905359, -0.3103738936138262], [-0.7237748022515089, -0.09977691527985856, -0.6369824102293615, -0.5175726287584196, 0.9211836260777865, 0.23797491976413193, -0.19754105014792864, -1.4349889655524184, -0.8966085298515106, -0.6369910539184891, 0.4017646537107139, -1.6777031029773297, 0.22319309675477164, 1.3716219738692008, 0.20822406864955134, -1.4136356236006473]]]], "dtype": "float64"}, {"__nd__": [3], "dtype": "int64"}], "e": {"__nd__": [[[[0.152531236410141, -0.4423615038394928, 0.635918915271759, -0.3231705129146576, 0.9160480499267578, -0.24178549647331238, 0.7277922630310059, 0.4247557818889618, 0.026690663769841194, -1.2192202806472778, -0.6102135181427002, 0.02077321894466877, 0.20200996100902557, -0.31290823221206665, 1.0698058605194092, -0.026882125064730644], [-0.16131548583507538, -0.1428021937608719, 0.2193591445684433, -0.6617012619972229, 1.2621572017669678, -0.5546873807907104, 0.9259778261184692, 0.36016109585762024, -0.5270131230354309, -0.6389829516410828, -0.7203505635261536, 0.16274580359458923, 0.5265436172485352, -0.5278929471969604, 0.24751122295856476, -0.5236474871635437], [-0.8004814386367798, -0.38562554121017456, 0.746901273727417, -0.8372583389282227, 0.6169648766517639, -0.5081605911254883, 0.5732828974723816, 0.0599295049905777, -0.1288173347711563, -0.5662515759468079, -0.7673212289810181, 0.7474312782287598, 0.3779059648513794, -0.11638909578323364, 1.1763319969177246, 0.038367655128240585]], [[0.4012831449508667, -0.8189517855644226, -0.1230241134762764, -0.4101629853248596, -0.9148399829864502, 0.44851937890052795, 0.5447099804878235, -0.7245714664459229, -0.4642069637775421, 0.4973078966140747, -0.5092936754226685, 0.10699682682752609, -0.5674424767494202, -0.3362087607383728, 0.07702548801898956, 0.2520658075809479], [0.7315952777862549, -1.3880839347839355, -0.31444087624549866, -0.6715927124023438, -0.191938117146492, 0.5841965079307556, -0.31966108083724976, -0.7415215373039246, -0.6906867027282715, 0.6638938784599304, -0.5583012700080872, -0.3953861594200134, -0.6739474534988403, -0.6018505692481995, 0.04914598539471626, 1.0404462814331055], [-0.10037677735090256, -1.3948395252227783, -0.3939201831817627, -0.7463249564170837, -0.6655865907669067, -0.1626407653093338, 1.2567152976989746, 0.04095916077494621, -1.1144943237304688, 1.2488380670547485, -1.2942944765090942, -0.2847781777381897, -0.9333863854408264, -0.9053117036819458, -0.2792316973209381, 0.7765495777130127]], [[-0.14553408324718475, -0.33513709902763367, -0.19291077554225922, 0.2713066637516022, -0.4775642156600952, -0.15387706458568573, -1.2144519090652466, 0.05698738247156143, -0.13973431289196014, 1.2767549753189087, -0.535409152507782, 0.6108254790306091, 0.022536013275384903, -1.2856040000915527, 0.22776177525520325, 0.4364694654941559], [-0.0008396419580094516, -0.14317910373210907, -0.19989413022994995, 0.4249421954154968, -0.31258851289749146, 0.013412260450422764, -1.3283398151397705, -0.31721535325050354, -0.3687836527824402, 1.0222114324569702, -0.44513002038002014, 0.6206355690956116, -0.009000085294246674, -1.3449714183807373, 0.058265481144189835, 0.5893276929855347], [-0.5356866717338562, -0.10165439546108246, 0.02556680701673031, -0.4883989691734314, 0.23983728885650635, 0.431983083486557, -0.9518799185752869, -1.382352352142334, 0.1787450611591339, 0.25420284271240234, -0.2367238700389862, 0.8579316735267639, -0.7368518710136414, -1.2248035669326782, -1.0713582038879395, 0.48498600721359253]], [[-0.5841983556747437, -0.49286603927612305, -0.47363796830177307, -0.7160943746566772, 0.7618924379348755, 0.28530991077423096, 0.2975499927997589, -1.1411051750183105, -1.3969395160675049, -0.5265073776245117, 0.782316267490387, -0.7214791178703308, -0.7726300954818726, 0.4978667199611664, -0.12200305610895157, -0.9429939985275269], [-0.43725207448005676, -0.49909478425979614, -0.5066105127334595, -0.742655336856842, 0.76817387342453, 0.3547332286834717, 0.20668713748455048, -1.2257280349731445, -1.1479871273040771, -0.5188354253768921, 0.7769440412521362, -0.895405113697052, -0.37184399366378784, 0.720655083656311, 0.015863720327615738, -1.0755748748779297], [0.008434798568487167, -0.7503867149353027, -0.4897693395614624, -0.9272335171699524, 0.6880295872688293, 0.5541249513626099, 0.2800840437412262, -1.2577378749847412, -0.8351792097091675, -0.4336227774620056, 0.9921389818191528, -0.7497850656509399, 0.017488183453679085, 0.7448774576187134, 0.15718971192836761, -1.119598627090454]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[9.913966072663378, 0.17262390495694396, 6.399972138463044, 9.502631927400104], [6.6278871640857755, -1.4186045865988168, -8.931775696729337, 22.415055372368148], [-8.269387073726993, 9.045872239304401, -9.062013881723749, 13.088943594061035]], [[19.591957479618138, 9.125440594729255, -7.1500516946395285, -3.152571810183678], [15.243004564039728, 15.398415862866928, 6.837772499436648, 2.7478544430465495], [16.31017802900672, 5.903589687397341, 3.886369253178093, 9.356130208175408]]], [[[4.2686840251759595, 12.40982690906087, 15.597609042356236, 5.54275517953724], [-23.182842846341508, 0.6133068325328407, -4.236564944668077, -11.071278936463905], [-4.453732475921068, 13.659889559437897, -13.975360289583108, -3.3094348607448163]], [[-6.956726512883405, -14.850434300850331, -14.81382082005304, 5.644017932726624], [-13.105068885039966, 18.737797825971104, -1.76421578880414, 3.6965552960626042], [5.7920363680826314, -15.607070178008824, 10.851687363431221, 1.834977743211895]]]], "dtype": "float64"}, {"__nd__": [[[[-0.5573485321062797, 5.815010668373903, 1.9042985698487105, 0.6299126594856252], [-12.268186115476286, -0.9675292561789444, 6.513888198344805, -7.9759998801994145], [8.178561688351255, -3.297694711675475, 5.654785968574355, 0.6295629503460909], [-10.139226898959958, -1.9818050425826073, -5.546740427849448, -8.010703620239358], [-8.306614971944228, -21.37591876877873, -3.1615431037227943, -1.5853561774254996], [-10.643060945171674, -16.113668214064546, 6.388094748110218, 3.449711856576938], [11.053841881236941, 10.607336061301444, -9.872229640913194, 0.06011100673613627]], [[-1.4337396642471854, 8.788109135951492, -9.638579321669535, -9.21253170062673], [12.514956929538378, -10.984589943839149, 1.0447307706046407, 1.7372960529276626], [-14.837582308147688, -7.096300578152896, -8.622818001556503, -5.376877262841616], [-10.777266133599197, 9.172459104505652, -3.2212248300863546, 9.326864873699648], [8.435156071180366, -17.412167489630082, 8.41761252720673, -5.435598662637032], [7.108392157623293, 9.810962785692853, 13.000451301290292, 9.913492101706456], [-1.2238753315073505, -5.905140400045223, 1.1537322188922758, 2.4891464549900593]]], [[[0.5770569254285681, -2.0709142748484415, 5.895617553159012, -3.8739392697423107], [2.7945200624010775, 1.6621372961790328, 11.289622126088814, -5.807067945717812], [-5.903431723728224, -8.522908712196742, 7.905972209154757, 3.994415274229459], [-2.5213379214895877, 17.319171949297278, -8.924501763704448, 8.430670023943334], [4.080509228916985, -3.8319615817989665, -10.46149621379371, -7.48110275720581], [8.56065215801702, -14.331548761627362, -2.7392515125936545, -8.372604666054343], [0.21118052689249894, -7.579536546772053, -4.881409850100765, -13.631590046348546]], [[1.5051446109465765, -8.639385740287226, 0.136235473322329, -17.913416011104093], [-12.790045625123188, -4.130018742789087, -7.94473514286553, -2.538964742388104], [-6.6293387228725305, 6.742764178768644, 4.666413934961699, 10.439782290442992], [6.179020031637757, 11.284162663788436, 11.188491608450963, 3.137950206895642], [-4.609452451530279, -0.741703500249744, 1.3502639666573941, -7.611386277676014], [-4.213728055133152, 10.140686522284781, 7.174562168758863, -7.676675671999723], [-9.818805242654324, -5.521361707571419, -16.505909763969104, 9.940154411550047]]]], "dtype": "float64"}, {"__nd__": [[[[0.4067111100061727, -1.1826721804649545, -0.28347319377123187, -0.7048225180508063], [1.2588847433115713, -1.2719426466999613, -0.7016236219726112, 1.5936876383647074], [1.651679637623774, 1.58198307761977, 0.339902533550405, -0.45507945914298825], [-0.8417765534820122, 0.7698848190968559, -0.7895463598421752, -0.11766523115944885], [-0.15935047922565262, 0.9994983221961601, -0.6867736863013597, 0.9862896505211084], [-0.5718997897395746, -0.4978431892596682, -1.1066347018203917, 0.24022511776988437], [-0.1639535610828244, 0.09595482384943738, -0.3228030797860971, 0.09821973563017525]], [[1.6490960940933383, 0.46774172148372145, 1.0111498549989666, -0.03908280913726877], [-0.019753397604080573, 1.180546316871672, 2.7069978603875215, -0.5614765277734042], [0.4950632296609845, -0.45810108123365373, 0.5397771645032526, 0.21497448216977313], [-0.5359531474742488, -1.5146708269674287, -1.594798471359023, 1.3328665951665422], [0.08029218251869223, 0.034501219807896936, 0.6044061833557958, -0.04382202565614805], [2.1327151696648, 0.16250830373374153, 1.0406491908033537, 0.22493454889800638], [-1.4269748750115094, 0.581437266179818, 0.30717013220262274, -0.4289567664377365]]], [[[1.4003562379252297, 0.6868408694359083, 0.8429736216688858, -1.1856246302144504], [-0.43678915000029495, -1.6480089814853178, 0.62639102267928, 1.0961965999095853], [-0.5624073484690868, 2.347591372057847, -1.056123813851711, 0.27564581874268235], [-2.0887799898564907, -1.9814150956964747, 0.3846695791974853, 0.14990582486747542], [2.0490009628492687, 0.26843612119690746, 1.3576086747468714, -0.7339189337880394], [0.6531334149965984, 1.4412438097258187, 0.12190188639986894, 1.4399828273616713], [1.1341078008034131, 0.31716122611250547, -0.9942716494144732, -0.5715850479953757]], [[-1.6239469066679413, 1.0230383717409315, 1.559780178840298, 0.3450878837549204], [0.46066383481022033, -1.9386208509814036, 1.148070372503899, -1.325418222839416], [-0.522591294762281, 2.0383995042010867, 0.5231010625328353, -0.7101239432651231], [-0.8089675845102837, 0.3188669729684601, 1.1807032326820024, 0.2648189374559386], [0.15744299938048592, 0.5361405360170438, 0.7515402186051914, -1.501982461402224], [-1.7881343899427238, 1.336923647731084, -0.6989886417570722, -2.1165791275656054], [-0.47635365626823084, -0.15916402148495748, -0.3215812745772684, 1.2122578455722945]]]], "dtype": "float64"}, {"__nd__": [4, 7], "dtype": "int64"}], "e": {"__nd__": [[[[1.6516796350479126, 1.5819830894470215, 0.33990252017974854, -0.45507946610450745], [1.6516796350479126, 1.5819830894470215, 0.33990252017974854, -0.45507946610450745], [0.4067111015319824, -1.182672142982483, -0.2834731936454773, -0.7048225402832031]], [[1.6488962173461914, 0.467827171087265, 1.0113531351089478, -0.03914540633559227], [-0.013875715434551239, 1.178035855293274, 2.7010252475738525, -0.559636652469635], [-0.01975339837372303, 1.1805462837219238, 2.706997871398926, -0.5614765286445618]]], [[[-0.4367891550064087, -1.648008942604065, 0.6263909935951233, 1.0961966514587402], [1.1341078281402588, 0.3171612322330475, -0.9942716360092163, -0.5715850591659546], [-2.088779926300049, -1.9814151525497437, 0.38466957211494446, 0.14990583062171936]], [[-0.47635364532470703, -0.15916402637958527, -0.32158127427101135, 1.2122578620910645], [-0.5225912928581238, 2.0383994579315186, 0.5231010913848877, -0.7101239562034607], [-1.6239469051361084, 1.023038387298584, 1.5597801208496094, 0.3450878858566284]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[-1.0972261531056364, -0.04261977114876756], [-1.4319315137258155, 2.0878134760311986]]], [[[-1.1723508723402303, -1.6459803605196246], [0.7118059496016084, 0.2203470883738987]]], [[[-1.2629014477980918, 0.5768149227733232], [0.7711194430055743, -0.43526035835571636]]], [[[0.5531688081070891, 0.9342617801938705], [0.6839355325478434, -0.47568652918019033]]]], "dtype": "float64"}, {"__nd__": [[[[-0.06457054172527496, -0.4592633009551297], [-0.4173046662128129, -0.06472206403386586]]], [[[-0.4638548055114337, -0.17068236577552573], [-0.1456434615918569, -0.22978985842130512]]], [[[0.5316944639978796, -1.1806266186226768], [-0.8719335077067029, -0.6028330417517209]]], [[[-0.8558589462898251, 0.4781332867507308], [0.23166213491383095, -0.06634406383986176]]]], "dtype": "float64"}, {"__nd__": [[[[-0.2757957283582708, 0.5284015797986005], [0.827677036476707, 0.8860006036689929]]], [[[-0.2270228140974382, 0.05842178100793766], [-1.6823109472149207, -0.24150313331195486]]], [[[2.1277526003195546, -0.7219653818299441], [0.9315422409630499, 0.6167035937011006]]], [[[-0.9663041042428302, -1.8166522541559023], [-0.8082622303501521, 1.27292743278853]]]], "dtype": "float64"}, {"__nd__": [1, 2, 1, 2], "dtype": "int64"}], "e": {"__nd__": [[[[-0.27579572796821594, 0.5284015536308289], [-0.27579572796821594, 0.5284015536308289]]], [[[-0.8839470744132996, -0.0769658088684082], [-1.0094830989837646, -0.10283792018890381]]], [[[2.1277525424957275, -0.7219653725624084], [2.1277525424957275, -0.7219653725624084]]], [[[-0.8846887350082397, -0.22114282846450806], [-0.8603847026824951, 0.2539794147014618]]]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(batched_attention, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")